# 07 — Three-Engine Evidence & Explain Demo

One schema, three engines, shared Store. All output comes from real framework calls.

**What this demonstrates:**
1. **Native** derivation → CandidateEvidenceTree with certainty
2. **ProbLog** derivation → CandidateEvidenceTree with proof_goal/proof_leaf + probability
3. **PyReason** propagation → CandidateProvenanceTimeline with chains + events
4. **Cross-engine data flow**: accepted facts from one engine feed the next
5. **Full explain pipeline** for each: tree/timeline → summary → narrative → NL

ProbLog and PyReason sections use mocked runners so the notebook runs without
external engine installs, but the entire evaluate → accept → explain pipeline is real.

**Prerequisites:** [01](01_sdk_basics.ipynb)–[06](06_problog_probabilistic.ipynb).

## 0. Imports

In [ ]:
import sys, tempfile, warnings
from pathlib import Path
from pprint import pprint
from unittest.mock import patch

sys.path.insert(0, str(Path(".").resolve().parent / "src"))

from factpy_kernel.sdk import SDKStore, Entity, Identity, Field, Rule, Pred, vars as sdk_vars
from factpy_kernel.sdk.compile import compile_schema_from_classes
from factpy_kernel.authoring import FileAuthoringRegistry
from factpy_kernel.core.evidence.write_protocol import set_field
from factpy_kernel.service.runtime_v1 import (
    open_runtime_session, close_runtime_session, reset_runtime_sessions_for_tests,
    write_runtime_fact, evaluate_runtime_derivation, accept_runtime_derivation,
    explain_runtime_tree, explain_runtime_summary, explain_runtime_narrative, explain_runtime_nl,
    explain_runtime_timeline, explain_runtime_timeline_summary, explain_runtime_timeline_narrative,
    export_runtime_package,
)
from factpy_kernel.audit import AuditQuery, load_audit_package

import factpy_kernel.adapters.problog
import factpy_kernel.adapters.pyreason
from factpy_kernel.adapters.pyreason.session import PyReasonSession
from factpy_kernel.adapters.pyreason.runner import PyReasonRunConfig, PyReasonRunResult

## 1. Shared Schema

A `Researcher` entity used by all three engines. Fields span the use cases:
- `expertise` / `impact_score`: seeded with `confidence` → drives **certainty** (native)
- `tag_seed` / `tag`: ProbLog reads seed → derives tag with **probability**
- `risk_flag`: PyReason propagates boolean **risk signal** across researchers

In [ ]:
class Researcher(Entity):
    researcher_id: str = Identity(primary_key=True)
    name: str = Field(cardinality="single")
    expertise: str = Field(cardinality="single")
    impact_score: str = Field(cardinality="single")
    tag_seed: str = Field(cardinality="single")
    tag: str = Field(cardinality="single")
    risk_flag: str = Field(cardinality="single")

schema_ir = compile_schema_from_classes([Researcher])
sdk = SDKStore([Researcher], schema_ir=schema_ir)
print(f"Schema: {len(schema_ir['predicates'])} predicates")
for p in schema_ir["predicates"]:
    print(f"  {p['pred_id']}")

## 2. Runtime Session + Seed Facts

Write facts **once** — all three engines will consume them from the same session.

In [ ]:
# Rule for native certainty
with sdk_vars("r", "exp", "score") as (r, exp, score):
    qualified_rule = Rule(
        id="q.qualified", version="1.0.0",
        select=[r, exp],
        where=[Pred("researcher:expertise", r, exp), Pred("researcher:impact_score", r, score)],
        expose=True,
        condition_weights={"b0.a0": 0.8, "b0.a1": 0.5},
    )

# Registry
registry_dir = tempfile.mkdtemp(prefix="three_engine_")
registry = FileAuthoringRegistry(Path(registry_dir))
registry.upsert_schema_ir(sdk.schema_ir)
registry.register_rule_spec(sdk._compile_rule_input(qualified_rule))

# Session
reset_runtime_sessions_for_tests()
resp = open_runtime_session({"registry_root": registry_dir})
session_id = resp["session"]["session_id"]

# Seed facts
alice_ref = sdk.ref(Researcher, researcher_id="Alice")
bob_ref = sdk.ref(Researcher, researcher_id="Bob")

facts = [
    ("researcher:name", alice_ref, [["string", "Alice Chen"]], {}),
    ("researcher:expertise", alice_ref, [["string", "NLP"]], {"confidence": 0.95}),
    ("researcher:impact_score", alice_ref, [["string", "92"]], {"confidence": 0.7}),
    ("researcher:tag_seed", alice_ref, [["string", "senior"]], {"confidence": 1.0}),
    ("researcher:name", bob_ref, [["string", "Bob Zhang"]], {}),
    ("researcher:expertise", bob_ref, [["string", "CV"]], {"confidence": 0.88}),
    ("researcher:impact_score", bob_ref, [["string", "85"]], {"confidence": 0.6}),
    ("researcher:tag_seed", bob_ref, [["string", "senior"]], {"confidence": 1.0}),
]
for pred_id, ref, terms, meta in facts:
    write_runtime_fact(session_id, {"pred_id": pred_id, "e_ref": ref,
        "rest_terms": terms, **({"meta": meta} if meta else {})}, kind="add")

print(f"Session {session_id[:20]}... ready")
print(f"  Alice: expertise=NLP(0.95), impact=92(0.7), tag_seed=senior")
print(f"  Bob:   expertise=CV(0.88),  impact=85(0.6), tag_seed=senior")

## 3. Engine A — Native Derivation (Certainty)

`condition_weights` + `confidence` → automatic certainty routing.
CandidateEvidenceTree with witness nodes and certainty summary.

In [ ]:
eval_native = evaluate_runtime_derivation(session_id, {"derivation": {
    "derivation_id": "drv.qualified", "version": "1.0.0",
    "target": "researcher:expertise", "head_vars": ["$r", "$exp"],
    "where": [["ruleref", "q.qualified", "1.0.0", ["$r", "$exp"]]],
    "mode": "native",
}})

native_cands = eval_native["evaluation"]["candidates"]
for c in native_cands:
    accept_runtime_derivation(session_id, {"candidate": c})

cid_native = native_cands[0]["candidate_id"]
print(f"Native: {len(native_cands)} candidates accepted")
print(f"  confidence_kind: {native_cands[0]['confidence_kind']}")

In [ ]:
# Full explain pipeline
tree = explain_runtime_tree(session_id, {"kind": "candidate", "id": cid_native})
summary = explain_runtime_summary(session_id, {"kind": "candidate", "id": cid_native})
narrative = explain_runtime_narrative(session_id, {"kind": "candidate", "id": cid_native})
nl = explain_runtime_nl(session_id, {"kind": "candidate", "id": cid_native})

print("=== Native: CandidateEvidenceTree ===")
print(f"tree.support_kind = {tree['tree']['support_kind']}")
print(f"tree.root.node_kind = {tree['tree']['root']['node_kind']}")

s = summary["summary"]
print(f"\nsummary.witness_assertion_count = {s['witness_assertion_count']}")
print(f"summary.rule_ref_count = {s['rule_ref_count']}")

cs = summary.get("certainty_summary")
if cs:
    print(f"\ncertainty_summary.aggregate_certainty = {cs['aggregate_certainty']}")
    print(f"certainty_summary.aggregation = {cs['aggregation']}")
    for c in cs["conditions"]:
        print(f"  {c['atom_key']}: weight={c['weight']}, impact={c['impact']}")

n = narrative["narrative"]
print(f"\nnarrative.headline = {n['headline']}")

print(f"\nNL ({len(nl['explain_nl']['paragraphs'])} paragraphs):")
for i, p in enumerate(nl["explain_nl"]["paragraphs"]):
    print(f"  [{i+1}] {p}")

## 4. Engine B — ProbLog Derivation (Probability)

ProbLog reads facts **already in the session** (same `tag_seed` facts seeded for native).
The runner is mocked, but evaluate → accept → explain is the real pipeline.

Returns CandidateEvidenceTree with `proof_goal`/`proof_leaf` nodes + `problog_probability`.

In [ ]:
# Mock ProbLog runner with realistic trace output
def _mock_problog_output(sdk_store):
    alice = sdk_store.ref(Researcher, researcher_id="Alice")
    return "\n".join([
        " call query(X1,X2) {0.00000} []",
        f'  result query(X1,X2) ("senior","{alice}") {{{{}}}} {{0.00012}} []',
        " complete query(X1,X2) {0.00013} {0.00013} []",
        f' call answer("senior","{alice}") {{0.00019}} [at 4:7]',
        f'  call researcher__tag_seed("senior","{alice}") {{0.00026}} [at 3:9]',
        f'   result researcher__tag_seed("senior","{alice}") ("senior","{alice}") {{{{}}}} {{0.00038}} [at 3:9]',
        f'  complete researcher__tag_seed("senior","{alice}") {{0.00039}} {{0.00013}} []',
        f'  result answer("senior","{alice}") ("senior","{alice}") {{{{}}}} {{0.00060}} []',
        f' complete answer("senior","{alice}") {{0.00061}} {{0.00042}} []',
        "",
        f'answer("senior","{alice}"):\t0.85',
    ])

with patch("factpy_kernel.adapters.problog.engine_eval.run_problog") as mock_run:
    mock_run.return_value = _mock_problog_output(sdk)

    eval_prob = evaluate_runtime_derivation(session_id, {"derivation": {
        "derivation_id": "drv.problog_tag", "version": "1.0.0",
        "target": "researcher:tag", "head_vars": ["$u", "$tag"],
        "where": [["pred", "researcher:tag_seed", ["$u", "$tag"]]],
        "mode": "problog",
    }})

prob_cand = eval_prob["evaluation"]["candidates"][0]
accept_runtime_derivation(session_id, {"candidate": prob_cand})
cid_prob = prob_cand["candidate_id"]

print(f"ProbLog: candidate accepted")
print(f"  support_kind = {prob_cand['support_kind']}")
print(f"  confidence = {prob_cand.get('confidence', 'N/A')}")

In [ ]:
# Full explain pipeline — real CandidateEvidenceTree with proof nodes
tree_p = explain_runtime_tree(session_id, {"kind": "candidate", "id": cid_prob})
summary_p = explain_runtime_summary(session_id, {"kind": "candidate", "id": cid_prob})
narrative_p = explain_runtime_narrative(session_id, {"kind": "candidate", "id": cid_prob})
nl_p = explain_runtime_nl(session_id, {"kind": "candidate", "id": cid_prob})

print("=== ProbLog: CandidateEvidenceTree (proof nodes) ===")
t = tree_p["tree"]
print(f"tree.support_kind = {t['support_kind']}")
print(f"tree.root.engine_meta.probability = {t['root']['engine_meta']['probability']}")

support_nodes = t["root"]["children"][0]["children"]
for node in support_nodes:
    kind = node["node_kind"]
    pred = node.get("pred_id", "?")
    args = node.get("goal_args", [])
    children_count = len(node.get("children", []))
    print(f"  {kind}: {pred}({', '.join(str(a) for a in args)}) [{children_count} children]")

sp = summary_p["summary"]
print(f"\nsummary.problog_probability = {sp.get('problog_probability')}")
print(f"summary.proof_goal_count = {sp.get('proof_goal_count')}")
print(f"summary.proof_leaf_count = {sp.get('proof_leaf_count')}")

np_ = narrative_p["narrative"]
print(f"\nnarrative.probability_lines = {np_.get('probability_lines')}")

print(f"\nNL ({len(nl_p['explain_nl']['paragraphs'])} paragraphs):")
for i, p in enumerate(nl_p["explain_nl"]["paragraphs"]):
    print(f"  [{i+1}] {p}")

## 5. Engine C — PyReason Propagation (Timeline)

PyReason reads facts from the **same session** — including facts accepted from native.
The runner is mocked with realistic event data, but the full pipeline is real.

Returns `CandidateProvenanceTimeline`, NOT CandidateEvidenceTree.

In [ ]:
# Mock PyReason runner with realistic propagation events
derived_session = PyReasonSession(schema_ir)
derived_session._write_node_fact_internal("researcher:risk_flag", alice_ref, "true", bound=[1.0, 1.0])

trace_dict = {
    "engine": "pyreason",
    "trace_type": "event_log",
    "timesteps": 2,
    "node_events": [
        {
            "time": 0, "fixpoint_op": 1,
            "component": alice_ref, "component_type": "node",
            "label": "risk_flag",
            "old_bound": [0.0, 1.0], "new_bound": [1.0, 1.0],
            "occurred_due_to": "seed_fact",
            "clause_groundings": [],
        },
    ],
    "edge_events": [],
}

with patch("factpy_kernel.adapters.pyreason.engine_eval.run_pyreason") as mock_pr:
    mock_pr.return_value = PyReasonRunResult(
        interpretation=None, trace=None, trace_dict=trace_dict,
        derived_session=derived_session,
        config=PyReasonRunConfig(timesteps=2, atom_trace=True),
        elapsed_seconds=0.01,
    )

    eval_pr = evaluate_runtime_derivation(session_id, {"derivation": {
        "derivation_id": "drv.pyreason_risk", "version": "1.0.0",
        "target": "researcher:risk_flag", "head_vars": ["$r"],
        "where": [["pred", "researcher:expertise", ["$r", "$exp"]]],
        "mode": "pyreason",
    }})

pr_cand = eval_pr["evaluation"]["candidates"][0]
accept_runtime_derivation(session_id, {"candidate": pr_cand})
cid_pr = pr_cand["candidate_id"]

print(f"PyReason: candidate accepted")
print(f"  support_kind = {pr_cand['support_kind']}")

In [ ]:
# explain-tree returns error (correct — PyReason is not tree)
tree_err = explain_runtime_tree(session_id, {"kind": "candidate", "id": cid_pr})
print(f"explain-tree: ok={tree_err['ok']} (PyReason uses timeline, not tree)")

# Timeline — real CandidateProvenanceTimeline
tl = explain_runtime_timeline(session_id, {"kind": "candidate", "id": cid_pr})
summary_pr = explain_runtime_timeline_summary(session_id, {"kind": "candidate", "id": cid_pr})
narrative_pr = explain_runtime_timeline_narrative(session_id, {"kind": "candidate", "id": cid_pr})
nl_pr = explain_runtime_nl(session_id, {"kind": "candidate", "id": cid_pr})

print(f"\n=== PyReason: CandidateProvenanceTimeline ===")
timeline = tl["timeline"]
print(f"timeline.kind = {timeline['kind']}")
print(f"timeline.timesteps = {timeline['timesteps']}")
print(f"timeline.chains ({len(timeline['chains'])}):")
for chain in timeline["chains"]:
    print(f"  {chain['component']}.{chain['label']}:")
    for evt in chain["events"]:
        print(f"    t={evt['time']}: {evt['old_bound']} → {evt['new_bound']} by {evt['trigger']}")

s = summary_pr["summary"]
print(f"\nsummary.explain_kind = {s['explain_kind']}")
print(f"summary.chain_count = {s['chain_count']}")
print(f"summary.seed_count = {s['seed_count']}")

n = narrative_pr["narrative"]
print(f"\nnarrative.headline = {n['headline']}")
for line in n.get("propagation_lines", []):
    print(f"  {line}")

print(f"\nNL kind = {nl_pr['kind']}")
for i, p in enumerate(nl_pr["explain_nl"]["paragraphs"]):
    print(f"  [{i+1}] {p}")

## 6. Cross-Engine Comparison

Three engines, same schema, same session — different explain surfaces:

In [ ]:
print("=" * 70)
print("CROSS-ENGINE EXPLAIN COMPARISON")
print("=" * 70)

# Native
s_native = explain_runtime_summary(session_id, {"kind": "candidate", "id": cid_native})["summary"]
cs = explain_runtime_summary(session_id, {"kind": "candidate", "id": cid_native}).get("certainty_summary")
print(f"\n[Native] {cid_native[:30]}...")
print(f"  contract: CandidateEvidenceTree")
print(f"  witnesses: {s_native['witness_assertion_count']}, rule_refs: {s_native['rule_ref_count']}")
if cs:
    print(f"  certainty: {cs['aggregate_certainty']} ({cs['aggregation']})")

# ProbLog
s_prob = explain_runtime_summary(session_id, {"kind": "candidate", "id": cid_prob})["summary"]
print(f"\n[ProbLog] {cid_prob[:30]}...")
print(f"  contract: CandidateEvidenceTree (proof nodes)")
print(f"  proof_goals: {s_prob.get('proof_goal_count', 0)}, proof_leaves: {s_prob.get('proof_leaf_count', 0)}")
print(f"  probability: {s_prob.get('problog_probability')}")

# PyReason
s_pr = explain_runtime_timeline_summary(session_id, {"kind": "candidate", "id": cid_pr})["summary"]
print(f"\n[PyReason] {cid_pr[:30]}...")
print(f"  contract: CandidateProvenanceTimeline")
print(f"  chains: {s_pr['chain_count']}, timesteps: {s_pr['timesteps']}")
print(f"  final_bound: {s_pr.get('final_bound')}")

print(f"\n{'=' * 70}")
print("All three consumed the same session facts. The Store is the integration bus.")
print(f"{'=' * 70}")

## 7. Cleanup

In [ ]:
close_runtime_session(session_id)
reset_runtime_sessions_for_tests()
print("Session closed.")

## Architecture Summary

```
  Shared Schema + Store (single session, single ledger)
       │
       ├── Native evaluate → accept → CandidateEvidenceTree
       │     witness nodes + certainty_summary
       │     explain-tree / explain-summary / explain-narrative / explain-nl
       │
       ├── ProbLog evaluate → accept → CandidateEvidenceTree  
       │     proof_goal / proof_leaf + problog_probability
       │     explain-tree / explain-summary / explain-narrative / explain-nl
       │
       └── PyReason evaluate → accept → CandidateProvenanceTimeline
             chains + events + bounds
             explain-timeline / explain-timeline-summary / explain-timeline-narrative
             + polymorphic dispatch on explain-summary / explain-narrative / explain-nl

  Cross-engine: accepted facts from Engine A are visible to Engine B
  Audit: evidence_graphs.jsonl (all) + provenance_timelines.jsonl (PyReason)
```